# I-Observe Face Pipeline (Colab)

Runs the same model family as the local app: **YOLO11 Face (5 real landmarks) → MiniFASNetV2 → aligned ArcFace 512-D ONNX → TurboVec cosine index**. Every detected face is handled independently. Upload licensed model weights and media; Colab storage is separate from the local project's database/index.

The ArcFace ONNX in this project is marked InsightFace non-commercial research-only. Confirm model and dataset rights before commercial deployment. The match threshold is configurable and still requires calibration on representative data.

## 1. Install and provide the same model artifacts

Upload `yolo11n-pose_widerface.pt`, `auraface_glintr100.onnx`, and `minifasnet_v2.onnx` from the project's `models/` directory when prompted. These are real models, not Colab-generated placeholders.

In [ ]:
!pip -q install ultralytics onnxruntime turbovec opencv-python-headless

import json, os, time
from pathlib import Path
import cv2, numpy as np
from google.colab import files
from ultralytics import YOLO
import onnxruntime as ort
from turbovec import IdMapIndex
ROOT=Path('/content/i-observe'); MODEL_DIR=ROOT/'models'; DATA_DIR=ROOT/'data'; MODEL_DIR.mkdir(parents=True,exist_ok=True); DATA_DIR.mkdir(parents=True,exist_ok=True)
MATCH_THRESHOLD=float(os.getenv('ARCFACE_MATCH_THRESHOLD','0.60'))
uploaded=files.upload()
for name,data in uploaded.items(): (MODEL_DIR/name).write_bytes(data)
required=['yolo11n-pose_widerface.pt','auraface_glintr100.onnx','minifasnet_v2.onnx']
missing=[n for n in required if not (MODEL_DIR/n).exists()]
if missing: raise FileNotFoundError(f'Upload required project model files: {missing}')
detector=YOLO(str(MODEL_DIR/required[0]))
fas=ort.InferenceSession(str(MODEL_DIR/required[2]),providers=['CPUExecutionProvider'])
arc=ort.InferenceSession(str(MODEL_DIR/required[1]),providers=['CPUExecutionProvider'])
print('Ready:', 'YOLO11 Face + MiniFASNetV2 + ArcFace 512-D + TurboVec; threshold=',MATCH_THRESHOLD)

In [ ]:
# Match the local project's provisional, data-derived operating point.
MATCH_THRESHOLD=float(os.getenv('ARCFACE_MATCH_THRESHOLD','0.25'))
LIVE_CLASS=1  # verified local MiniFASNetV2 class contract
def live_score(face_crop):
    if face_crop.size==0: raise ValueError('Empty MiniFASNet crop')
    resized=cv2.resize(face_crop,(80,80),interpolation=cv2.INTER_LINEAR)
    inp=np.ascontiguousarray(resized.transpose(2,0,1)[None].astype(np.float32))  # raw BGR 0..255
    inputs=fas.get_inputs(); logits=fas.run(None,{inputs[0].name:inp})[0].reshape(-1).astype(np.float64)
    probs=np.exp(logits-logits.max()); probs/=probs.sum()
    if LIVE_CLASS>=len(probs): raise RuntimeError('MiniFASNet live class index does not match output')
    return float(probs[LIVE_CLASS])


## Stage 1–3: detection, alignment, preprocessing

In [ ]:
def detect(frame):
    result=detector.predict(frame, imgsz=640, conf=.25, max_det=500, verbose=False)[0]
    if result.keypoints is None: return []
    boxes=result.boxes.xyxy.cpu().numpy(); points=result.keypoints.xy.cpu().numpy(); scores=result.boxes.conf.cpu().numpy()
    return [(b.astype(float),p.astype(float),float(s)) for b,p,s in zip(boxes,points,scores)]

def align_arcface(frame, points, size=112):
    # Five detector-provided landmarks: right eye, left eye, nose, right mouth, left mouth.
    dst=np.array([[38.2946,51.6963],[73.5318,51.5014],[56.0252,71.7366],[41.5493,92.3655],[70.7299,92.2041]],np.float32)
    matrix,_=cv2.estimateAffinePartial2D(points.astype(np.float32),dst,method=cv2.LMEDS)
    if matrix is None: raise ValueError('Five-point affine alignment failed')
    aligned=cv2.warpAffine(frame,matrix,(size,size),borderValue=0)
    inp=((aligned[:,:,::-1].astype(np.float32)-127.5)/127.5).transpose(2,0,1)[None]
    name=arc.get_inputs()[0].name; emb=arc.run(None,{name:inp})[0].reshape(-1).astype(np.float32)
    if emb.size!=512 or not np.isfinite(emb).all(): raise ValueError(f'Expected finite 512-D ArcFace embedding, got {emb.shape}')
    norm=float(np.linalg.norm(emb));
    if norm<1e-12: raise ValueError('Zero ArcFace embedding')
    return aligned,emb/norm

def live_score(face_crop):
    inp=cv2.resize(face_crop,(80,80)).astype(np.float32).transpose(2,0,1)[None]
    inputs=fas.get_inputs(); raw=fas.run(None,{inputs[0].name:inp})[0].reshape(-1)
    probs=np.exp(raw-raw.max()); probs/=probs.sum()
    return float(probs[-1])


## 2. Real MiniFASNet + 512-D TurboVec enrollment/search

No motion heuristic, SFace, NumPy gallery, or Qdrant fallback is used in this notebook.

In [ ]:
INDEX_PATH=DATA_DIR/'face_index.tvim'; META=DATA_DIR/'identities.json'
if INDEX_PATH.exists()!=META.exists(): raise RuntimeError('TurboVec index/metadata mismatch; refusing fallback')
index=IdMapIndex.load(str(INDEX_PATH)) if INDEX_PATH.exists() else IdMapIndex(dim=512,bit_width=4)
identities=json.loads(META.read_text()) if META.exists() else {'next_id':1,'ids':{}}
def persist(): index.sync(str(INDEX_PATH)); META.write_text(json.dumps(identities),encoding='utf-8')
def add_identity(name,vector):
    if np.asarray(vector).size!=512: raise ValueError('Enrollment requires a 512-D vector')
    pid=identities['next_id']; index.add_with_ids(np.asarray(vector,np.float32).reshape(1,512),np.asarray([pid],np.uint64)); identities['ids'][str(pid)]=name; identities['next_id']+=1; persist()
def search(vector):
    if len(index)==0: return {'identity':'unknown','score':0.0}
    scores,ids=index.search(np.asarray(vector,np.float32).reshape(1,512),k=1); pid=str(int(ids[0,0])); score=float(np.clip(scores[0,0],-1,1)); name=identities['ids'].get(pid)
    if name is None: raise RuntimeError('TurboVec returned an unmapped point id')
    return {'identity':name if score>=MATCH_THRESHOLD else 'unknown','score':score}


## 3. Run one uploaded image through all faces

Liveness scores are per face. Enroll only consented, known-live images; a still photo is not proof of liveness. For each valid live face, this cell runs affine alignment, ArcFace and TurboVec lookup independently.

In [ ]:
from google.colab import files
media=files.upload(); media_path=next(iter(media)); frame=cv2.imread(media_path)
if frame is None: raise RuntimeError('Could not decode uploaded image')
faces=detect(frame); print('detected_faces=',len(faces))
for bbox,points,confidence in faces:
    x1,y1,x2,y2=np.rint(bbox).astype(int); crop=frame[max(0,y1):min(frame.shape[0],y2),max(0,x1):min(frame.shape[1],x2)]
    try:
        live=live_score(crop); aligned,emb=align_arcface(frame,points)
        result=search(emb) if live>=0.5 else {'identity':'spoof_rejected','score':live}
    except Exception as exc: result={'identity':'processing_error','score':0.0}; print('face error:',exc)
    cv2.rectangle(frame,(x1,y1),(x2,y2),(0,220,0),2); cv2.putText(frame,f"{result['identity']} {result['score']:.3f}",(x1,max(20,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,.6,(0,220,0),2)
from IPython.display import display
display(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))

## Quality checks and limitations

- Five keypoints must come from the YOLO11 face checkpoint; missing landmarks fail alignment rather than being fabricated.
- Each detected face has isolated processing/error handling.
- TurboVec persists only inside this Colab runtime unless you explicitly mount durable Drive storage. It does not connect to the local app's TurboVec files, PostgreSQL, evidence storage, or UI.
- The 0.60 match threshold is configurable but not a validated biometric operating point; calibrate false-accept/false-reject rates before any consequential use.
- MiniFASNet's model-specific input preprocessing/class ordering must match its export. Confirm against known live/spoof validation samples before trusting liveness scores.
- The local application remains the integrated system of record; this notebook is an isolated reproducible model experiment, not a remote extension of the running app.